# Lecture 6. Simple linear regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture06/lecture06.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 11.1 to 11.6.

This notebook follows the lecture slides. The two methods of today, the regression line and the test of its slope,
are worked through in the same steps as before: the question, what we know and want to find out, the formula, the
calculation by hand (on the slides), the same calculation in Python, and the answer for a manager. In Python we first
follow the formula line by line and then fit the model with statsmodels. Where Python does something differently
from Newbold, a section "... in Python and in Newbold" says so.

Parts 1 to 4 each end with an exercise. In an exercise you predict a result, run the cell, change one value, and
answer a question about what you see. You do not need to write any code. The last section holds a summary table,
the symbols and the Danish terms, so you can use the notebook as a reference afterwards.

Run the setup cell first. It loads the RetailCorp customer data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 customers drawn at random

print(f"Customers: {len(customers):,}")
customers.head()

## The case and the question

RetailCorp's population of customers is the 60,000 customers who had an account before 1 January 2025 and ordered at
least once in 2025. The analysts drew a random sample of 1,000. Today we use two columns:

| Column | Content |
|---|---|
| `tenure_years` | Years since the customer's first order, on 1 January 2025 (x) |
| `spend_2025` | Total spend in DKK in 2025 (y) |

The other columns (age, region, channel, club membership, visits, churn) come back in Lectures 7 to 9. Exercises 1
and 2 use `age` and `visits_2025`.

**Lead question.** Do customers who have been with RetailCorp longer spend more per year, and how much more?

In Lecture 4 the explanatory variable had two values (Copenhagen or Aarhus), and we compared two averages. Today x
can take many values, so we describe how average spend changes with x by a straight line.

In [ ]:
customers[["tenure_years", "spend_2025"]].describe().round(2)

## Part 1. The line

**Question.** Do customers who have been with us longer spend more per year, and how much more?

**What we know.** 1,000 customers drawn at random; for each, years as a customer (x) and spend in 2025 (y).

**What we want.** The straight line that describes how average spend changes with years as a customer: its slope
b<sub>1</sub> (DKK more per extra year) and its intercept b<sub>0</sub> (the spend at 0 years).

First the scatter plot: one dot per customer.

In [ ]:
x = customers["tenure_years"]
y = customers["spend_2025"]

plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=8, alpha=0.5)
plt.xlabel("Years as a customer")
plt.ylabel("Spend in 2025, DKK")
plt.show()

**The model.** In the population, the average spend is a straight line in x, and each customer deviates from it:
y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub>x<sub>i</sub> + ε<sub>i</sub>. From the sample we estimate the line
ŷ<sub>i</sub> = b<sub>0</sub> + b<sub>1</sub>x<sub>i</sub>. The residual is e<sub>i</sub> = y<sub>i</sub> −
ŷ<sub>i</sub>, and least squares chooses b<sub>0</sub> and b<sub>1</sub> so that the sum of squared residuals,
SSE, is as small as possible. The formulas:

b<sub>1</sub> = Σ(x<sub>i</sub> − x̄)(y<sub>i</sub> − ȳ) / Σ(x<sub>i</sub> − x̄)²   and   b<sub>0</sub> = ȳ −
b<sub>1</sub>x̄

**The line in Python**, first by the formula, then with statsmodels.

In [ ]:
sxy = ((x - x.mean()) * (y - y.mean())).sum()   # Σ(x − x̄)(y − ȳ)
sxx = ((x - x.mean())**2).sum()                  # Σ(x − x̄)²
b1 = sxy / sxx
b0 = y.mean() - b1 * x.mean()
print(f"x-bar = {x.mean():.4f}, y-bar = {y.mean():.3f}, sxy = {sxy:.2f}, sxx = {sxx:.2f}")
print(f"b0 = {b0:.2f}, b1 = {b1:.3f}")

model = smf.ols("spend_2025 ~ tenure_years", data=customers).fit()
print(model.params)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=8, alpha=0.5)
xs = np.array([0, 12])
plt.plot(xs, b0 + b1 * xs, color="darkorange", linewidth=2.5)
plt.xlabel("Years as a customer")
plt.ylabel("Spend in 2025, DKK")
plt.show()

**For the manager.** Customers one year apart in tenure differ in average spend in 2025 by about 200 DKK. The line
puts the spend of a customer who had just joined at about 1,500 DKK. This compares customers as they are and does not
show that one more year raises a customer's spend (Part 4).

**The intercept.** b<sub>0</sub> is the line's value at x = 0: here, a customer who joined just before 2025. The
cell below counts the customers close to 0 years, so x = 0 lies within the data. When there are no data near x = 0, the intercept is an
extrapolation (Exercise 1).

In [ ]:
print("Customers with less than half a year:", (x < 0.5).sum())

### Fitting a line in Python and in Newbold

- Newbold's b<sub>0</sub> and b<sub>1</sub> are the column `coef` in the output, rows `Intercept` and `tenure_years`.
- `smf.ols("y ~ x", data=...)` adds the intercept by itself. `sm.OLS(y, x)` does not: without `sm.add_constant(x)`
  it forces the line through 0. This is a common error in code from the internet and from LLMs. The cell shows the
  difference.
- `np.polyfit(x, y, 1)` and `stats.linregress(x, y)` give the same b<sub>0</sub> and b<sub>1</sub>, but not the
  output table. Use them only as a check.
- `smf.ols` drops rows with a missing x or y without a warning. Check `No. Observations` in the output.

In [ ]:
through_zero = sm.OLS(y, x).fit()                  # no intercept: the line is forced through 0
with_constant = sm.OLS(y, sm.add_constant(x)).fit()
print("Without a constant: slope", round(through_zero.params.iloc[0], 2))
print("With a constant:   ", with_constant.params.round(3).to_dict())
print("np.polyfit:        ", np.polyfit(x, y, 1).round(3))

### Demonstration: is this the best line?

The cell tries lines through the point (x̄, ȳ) with other slopes and adds up the squared residuals for each. The
smallest SSE comes at the least-squares slope.

In [ ]:
slopes = [0, 100, 150, b1, 250, 300]
for b in slopes:
    a = y.mean() - b * x.mean()          # the line passes through (x-bar, y-bar)
    resid = y - (a + b * x)
    sse = (resid**2).sum()
    print(f"slope {b:6.1f}   intercept {a:8.1f}   SSE {sse / 1e9:.4f} billion")

grid = np.arange(0, 401, 10)
sse_grid = []
for b in grid:
    a = y.mean() - b * x.mean()
    sse_grid.append(((y - (a + b * x))**2).sum() / 1e9)
plt.figure(figsize=(6, 3.5))
plt.plot(grid, sse_grid)
plt.scatter([b1], [((y - (b0 + b1 * x))**2).sum() / 1e9], color="darkorange", zorder=3)
plt.xlabel("Slope of the line (DKK per year)")
plt.ylabel("SSE (billion)")
plt.show()

### Exercise 1. Age instead of years as a customer

**Predict.** The cell fits the line for spend against age. Will the slope be positive? What will the line say about
spend at age 0?

**Run** the cell. It prints b<sub>0</sub>, b<sub>1</sub>, R² and the smallest x in the data.

In [ ]:
x_var = "age"      # change to "tenure_years" in step 3

m1 = smf.ols(f"spend_2025 ~ {x_var}", data=customers).fit()
print(f"x = {x_var}: b0 = {m1.params.iloc[0]:.2f}, b1 = {m1.params.iloc[1]:.3f}, R2 = {m1.rsquared:.3f}, "
      f"smallest x = {customers[x_var].min()}")

**Change** `x_var = "age"` to `"tenure_years"` and run the cell again.

**Answer.** For which of the two lines does the intercept describe real customers? Why?

## Part 2. How well the line fits

Each customer's distance from the average spend ȳ splits into the part along the line (what tenure accounts for) and
the residual (what is left). Squared and added up over all customers:

- SST = Σ(y<sub>i</sub> − ȳ)², the total variation
- SSR = Σ(ŷ<sub>i</sub> − ȳ)², the part the line accounts for
- SSE = Σ(y<sub>i</sub> − ŷ<sub>i</sub>)², the part left in the residuals

SST = SSR + SSE, and R² = SSR / SST = 1 − SSE / SST. The typical distance of a customer from the line is
s<sub>e</sub> = √(SSE / (n − 2)): we divide by n − 2 because two numbers, b<sub>0</sub> and b<sub>1</sub>, were
estimated from the data.

In [ ]:
n = len(customers)
y_hat = b0 + b1 * x
sst = ((y - y.mean())**2).sum()
ssr = ((y_hat - y.mean())**2).sum()
sse = ((y - y_hat)**2).sum()
r2 = ssr / sst
s_e = np.sqrt(sse / (n - 2))
r = np.corrcoef(x, y)[0, 1]
print(f"SST = {sst / 1e6:.1f}, SSR = {ssr / 1e6:.1f}, SSE = {sse / 1e6:.1f} (million)")
print(f"R2 = {r2:.4f}, r = {r:.4f}, r squared = {r**2:.4f}")
print(f"s_e = {s_e:.2f} DKK; standard deviation of spend {y.std(ddof=1):.2f} DKK")
print(f"statsmodels: R2 = {model.rsquared:.4f}, s_e = {np.sqrt(model.scale):.2f}")

**For the manager.** Years as a customer account for 13% of the variation in spend; the other 87% is not accounted
for by tenure. The typical distance of a customer from the line is about 1,393 DKK.

R² does not tell you whether x causes y, whether a straight line is the right shape (Lecture 7), or whether the slope
is significant (Exercise 2). There is no threshold for a good R².

### Exercise 2. Visits instead of years as a customer

**Predict.** The cell fits spend against the number of visits in 2025. Will R² be higher or lower than 0.133?

**Run** the cell. It prints b<sub>1</sub>, its p-value, R² and s<sub>e</sub>.

In [ ]:
x_var = "visits_2025"      # change to "tenure_years" in step 3

m2 = smf.ols(f"spend_2025 ~ {x_var}", data=customers).fit()
print(f"x = {x_var}: b1 = {m2.params.iloc[1]:.3f}, p = {m2.pvalues.iloc[1]:.1e}, R2 = {m2.rsquared:.3f}, "
      f"s_e = {np.sqrt(m2.scale):.0f}")

**Change** `x_var = "visits_2025"` to `"tenure_years"` and run the cell again.

**Answer.** How can the slope for visits be clearly different from 0 while visits account for so little of the
variation?

## Part 3. Testing the slope

b<sub>1</sub> is computed from 1,000 customers drawn at random. Another sample would give another b<sub>1</sub>,
just as x̄ varied from sample to sample in Lecture 1. The standard error s<sub>b1</sub> measures how much.

**Question.** Is there a relationship between years as a customer and spend among all 60,000 customers, or could the
slope in the sample be chance?

**What we want.** H<sub>0</sub>: β<sub>1</sub> = 0 against H<sub>1</sub>: β<sub>1</sub> ≠ 0 at α = 0.05, and a
95% confidence interval for β<sub>1</sub>. The test is two-sided, as in the statsmodels output. The marketing question
asks about higher spend, so a one-sided H<sub>1</sub>: β<sub>1</sub> > 0 would also be defensible; for a one-sided
test, halve the p-value when b<sub>1</sub> has the sign in H<sub>1</sub>.

**The formulas.** s<sub>b1</sub> = s<sub>e</sub> / √Σ(x<sub>i</sub> − x̄)², t = b<sub>1</sub> / s<sub>b1</sub>
with n − 2 degrees of freedom, and the interval b<sub>1</sub> ± t<sub>n−2, α/2</sub> s<sub>b1</sub>.

**What the test assumes.** The average of y is a straight line in x; customers are independent; the scatter around
the line is the same for all x; the errors are normally distributed (for small samples). In these data the scatter
grows with tenure, and the residuals are skewed. Lecture 7 checks the assumptions with the residuals.

In [ ]:
s_b1 = s_e / np.sqrt(sxx)                         # s_b1
t_stat = b1 / s_b1
t_crit = stats.t.ppf(0.975, df=n - 2)
p = 2 * stats.t.sf(abs(t_stat), df=n - 2)         # two-sided
print(f"s_b1 = {s_b1:.3f}, t = {t_stat:.3f}, p = {p:.2e}")
print(f"crit = {t_crit:.4f}")
print(f"{b1 - t_crit*s_b1:.2f} to {b1 + t_crit*s_b1:.2f}")

The same numbers are in the statsmodels output, row `tenure_years`: `std err`, `t`, `P>|t|` and the interval
`[0.025  0.975]`.

In [ ]:
print(model.summary())

In [ ]:
# the same numbers one by one
print(model.bse["tenure_years"], model.tvalues["tenure_years"], model.pvalues["tenure_years"])
print(model.conf_int().loc["tenure_years"].values)
print("F =", round(model.fvalue, 2), " t squared =", round(model.tvalues["tenure_years"]**2, 2))

**For the manager.** If tenure and spend were unrelated among all customers, a sample slope this far from 0 would
almost never occur (p < 0.001). With 95% confidence, customers one year apart in tenure differ in average spend by
about 170 to 234 DKK. These standard errors assume the same scatter around the line for all x; Lecture 7 checks that.

### Testing the slope in Python and in Newbold

- s<sub>b1</sub> is `std err`; the interval is `[0.025  0.975]`.
- `P>|t|` tests H<sub>0</sub>: β<sub>1</sub> = 0 against β<sub>1</sub> ≠ 0. For a one-sided H<sub>1</sub>, halve
  the p-value if b<sub>1</sub> has the sign in H<sub>1</sub>.
- The interval uses the t value with n − 2 = 998 degrees of freedom, 1.9623. The normal value 1.96 is close
  here; with small samples the difference grows.
- A p-value printed as 0.000 means below 0.0005; `model.pvalues` gives the exact value. Report p < 0.001.
- With one x, the F test in the output is the same test as the t test: F = t², with the same p-value.

### Exercise 3. Fewer customers

**Predict.** The cell uses only the first n = 50 customers. What happens to b<sub>1</sub>, to its standard error and
to the interval?

**Run** the cell. It prints b<sub>1</sub>, s<sub>b1</sub>, t, p and the interval.

In [ ]:
n_used = 50      # change to 100, then to 1000, in step 3

m3 = smf.ols("spend_2025 ~ tenure_years", data=customers.head(n_used)).fit()
lo, hi = m3.conf_int().loc["tenure_years"]
print(f"n = {n_used}: b1 = {m3.params['tenure_years']:.1f}, s_b1 = {m3.bse['tenure_years']:.1f}, "
      f"t = {m3.tvalues['tenure_years']:.2f}, p = {m3.pvalues['tenure_years']:.2e}, interval {lo:.1f} to {hi:.1f}")

**Change** `n_used = 50` to `100`, then to `1000`, and run the cell again each time.

**Answer.** With 50 customers the slope is still significant. What can a manager say from the interval?

## Part 4. Prediction and the output

**Prediction.** The line predicts the average spend of customers with a given number of years. A single customer can
be far from it (s<sub>e</sub> = 1,393 DKK). Tenure in the data runs to 11.8 years, so a prediction for 20 years
is an extrapolation: the data cannot tell whether the line goes on.

In [ ]:
new = pd.DataFrame({"tenure_years": [5, 20]})
print(model.predict(new))
print("Largest tenure in the data:", x.max())

**Reading the output.** Covered today: `Dep. Variable` (y), `No. Observations` (n, after rows with missing values are
dropped), `Df Residuals` (n − 2), `R-squared`, `F-statistic` and `Prob (F-statistic)` (the slope test again), and the
coefficient table: `coef` (b<sub>0</sub> and b<sub>1</sub>), `std err` (s<sub>b1</sub>), `t`, `P>|t|` and the 95%
interval. The output does not print s<sub>e</sub>; use `np.sqrt(model.scale)`.

Later: `Adj. R-squared`, the F test with several x, `Covariance Type` (robust standard errors), `Omnibus`,
`Jarque-Bera`, `Skew` and `Kurtosis` (are the residuals normal?), `Durbin-Watson` (meaningful only when the rows are
in time order) and `Cond. No.` (closely related explanatory variables) in Lecture 7; `Log-Likelihood` returns with
logit in Lecture 9. `AIC` and `BIC` are not used in this course.

**Association and cause.** Customers with more years spend more. Three explanations give the same b<sub>1</sub>:
tenure raises spend (customers get to know the range); spend keeps customers (those who buy a lot stay, the others
leave and are not in the data); or a third variable such as age (older customers have been customers longer and also
spend more). The regression describes the customers as they are. Lecture 7 adds age to the model.

### Exercise 4. Reading the output for Web customers

**Read** b<sub>1</sub>, s<sub>b1</sub>, t, the p-value, the interval and R² for the Web customers in the output
the cell prints (it is also on the slide). **Check** whether F equals t².

**Predict** by hand the spend of a Web customer with five years, from b<sub>0</sub> and b<sub>1</sub>. **Run** the
cell to check your prediction.

In [ ]:
years = 5      # change to 10 in step 3

m4 = smf.ols("spend_2025 ~ tenure_years", data=customers[customers["channel"] == "Web"]).fit()
print(m4.summary())
print(f"Predicted spend at {years} years:", round(m4.predict(pd.DataFrame({"tenure_years": [years]})).iloc[0], 1))

**Change** `years = 5` to `10` and run the cell again.

**Answer.** Write the result for the Web customers in two sentences for a manager.

## Reporting a regression

We regressed spend in 2025 on years as a customer for a random sample of 1,000 customers (OLS). Customers one year
apart in tenure differ in average spend by 202 DKK (95% interval 170 to 234; t = 12.37, p < 0.001). Tenure accounts
for 13% of the variation in spend (R² = 0.133), and the typical distance of a customer from the line is 1,393 DKK.
The line predicts 2,504 DKK for customers with five years. The standard errors assume the same scatter around the
line for all x, which Lecture 7 checks. The data describe customers as they are and do not show that tenure raises
spend.

Report: what was regressed on what, the method and n; the slope with its unit and its interval; the test result and
R²; predictions only inside the data; the result described as an association.

## Reference

### Simple regression in one table

| | Quantity | Formula | RetailCorp customers | In the output |
|---|---|---|---|---|
| b<sub>1</sub> | Slope | Σ(x − x̄)(y − ȳ) / Σ(x − x̄)² | 201.785 DKK per year | `coef`, row `tenure_years` |
| b<sub>0</sub> | Intercept | ȳ − b<sub>1</sub>x̄ | 1,494.70 DKK | `coef`, row `Intercept` |
| R² | Share of the variation accounted for | SSR / SST = 1 − SSE / SST | 0.133 | `R-squared` |
| s<sub>e</sub> | Typical distance from the line | √(SSE / (n − 2)) | 1,393.14 DKK | not printed: `np.sqrt(model.scale)` |
| s<sub>b1</sub> | Standard error of the slope | s<sub>e</sub> / √Σ(x − x̄)² | 16.319 | `std err` |
| t | Test of H<sub>0</sub>: β<sub>1</sub> = 0 | b<sub>1</sub> / s<sub>b1</sub>, n − 2 df | 12.365, p < 0.001 | `t`, `P>\|t\|` |
| | 95% interval for β<sub>1</sub> | b<sub>1</sub> ± t<sub>n−2, 0.025</sub> s<sub>b1</sub> | 169.76 to 233.81 | `[0.025  0.975]` |
| ŷ | Prediction at x = 5 | b<sub>0</sub> + b<sub>1</sub> × 5 | 2,503.62 DKK | `model.predict(...)` |

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| β<sub>0</sub>, β<sub>1</sub> | beta-zero, beta-one | Intercept and slope of the population line (unknown) |
| ε<sub>i</sub> | epsilon-i | Customer i's deviation from the population line |
| b<sub>0</sub>, b<sub>1</sub> | b-zero, b-one | Intercept and slope estimated from the sample |
| ŷ<sub>i</sub> | y-hat-i | The line's value for customer i: the predicted y |
| e<sub>i</sub> | e-i | Residual: y<sub>i</sub> − ŷ<sub>i</sub> |
| SST, SSR, SSE | | Total, regression and error sums of squares |
| s<sub>e</sub> | s-e | Standard error of the estimate: the typical distance from the line |
| s<sub>b1</sub> | s-b-one | Standard error of the slope |

### Danish terms

| English | Danish |
|---|---|
| regression analysis | regressionsanalyse |
| dependent variable, explanatory variable | afhængig variabel, forklarende variabel |
| intercept, slope | skæring (konstantled), hældning |
| residual | residual |
| least squares | mindste kvadraters metode |
| coefficient of determination (R²) | forklaringsgrad |
| sum of squares | kvadratsum |
| standard error of the estimate | residualspredning |
| standard error of the slope | standardfejl for hældningen |
| regression coefficient | regressionskoefficient |
| prediction, extrapolation | prædiktion, ekstrapolation |